# 3. China A-shares: SSE Composite and CSI 300

Same real-time method applied to A-share index returns from AKShare (60-month warm-up, since the history is short).

Caveats: these are price indices (no dividends) and returns are not in excess of the risk-free rate.

In [1]:
import pandas as pd

from data_loaders import load_index
from vol_tools import (to_monthly, realized_variance, vol_managed_rt,
                       spanning_regression, sharpe, net_of_cost, vol_buckets)

In [2]:
indices = {"SSE": "sh000001", "CSI300": "sh000300"}

f, rv = {}, {}
for name, symbol in indices.items():
    d = load_index(symbol)
    f[name] = to_monthly(d)
    rv[name] = realized_variance(d)
    print(f"{name}: {d.index[0].date()} -> {d.index[-1].date()}")

SSE: 1990-12-20 -> 2026-10-30
CSI300: 2002-01-07 -> 2026-10-30


## 3.1 Real-time volatility management (cap = 1 means no leverage)

In [3]:
for name in indices:
    print(f"===== {name} =====")
    for cap in [1, 2]:
        fs, w = vol_managed_rt(f[name], rv[name], warmup=60, cap=cap)
        r = spanning_regression(fs, f[name])
        print(f"cap={cap}: from {fs.index[0]} | Sharpe orig {sharpe(f[name].loc[fs.index]):.2f} -> "
              f"managed {sharpe(fs):.2f} (net of 0.1% cost {sharpe(net_of_cost(fs, w, 0.001)):.2f}) | "
              f"alpha {r['alpha']:.2f} (t {r['alpha'] / r['alpha_se']:.2f}) | appraisal {r['appraisal']:.2f}")

===== SSE =====
cap=1: from 1996-01 | Sharpe orig 0.37 -> managed 0.37 (net of 0.1% cost 0.35) | alpha 0.48 (t 0.97) | appraisal 0.17
cap=2: from 1996-01 | Sharpe orig 0.37 -> managed 0.36 (net of 0.1% cost 0.34) | alpha 0.47 (t 0.91) | appraisal 0.16
===== CSI300 =====
cap=1: from 2007-02 | Sharpe orig 0.23 -> managed 0.32 (net of 0.1% cost 0.31) | alpha 2.31 (t 1.13) | appraisal 0.25
cap=2: from 2007-02 | Sharpe orig 0.23 -> managed 0.37 (net of 0.1% cost 0.35) | alpha 4.81 (t 1.37) | appraisal 0.30


## 3.2 Same months for both indices

The CSI 300 results start around 2010, the SSE results around 1996. Comparing them over the same months
separates the effect of the index from the effect of the time period.

In [4]:
fs_sse, w_sse = vol_managed_rt(f["SSE"], rv["SSE"], warmup=60, cap=1)
fs_csi, _ = vol_managed_rt(f["CSI300"], rv["CSI300"], warmup=60, cap=1)
same = fs_csi.index

print(f"SSE    (same months): orig {sharpe(f['SSE'].loc[same]):.2f} -> managed {sharpe(fs_sse.loc[same]):.2f}")
print(f"CSI300 (same months): orig {sharpe(f['CSI300'].loc[same]):.2f} -> managed {sharpe(fs_csi):.2f}")

SSE    (same months): orig 0.18 -> managed 0.38
CSI300 (same months): orig 0.23 -> managed 0.32


## 3.3 The paper's premise in A-shares

In [5]:
for name in indices:
    print(name)
    print(vol_buckets(f[name], rv[name]).round(3), "\n")

print("SSE 1996-2009 (the period where volatility timing did not help)")
print(vol_buckets(f["SSE"].loc["1996":"2009"], rv["SSE"]).round(3), "\n")
print("SSE 2010 onward")
print(vol_buckets(f["SSE"].loc["2010":], rv["SSE"]).round(3))

SSE
         mean    std  sharpe
bucket                      
low     0.172  0.225   0.766
mid     0.215  0.566   0.380
high    0.148  0.596   0.248 

CSI300
         mean    std  sharpe
bucket                      
low     0.104  0.182   0.572
mid     0.151  0.263   0.574
high   -0.009  0.322  -0.027 

SSE 1996-2009 (the period where volatility timing did not help)
         mean    std  sharpe
bucket                      
low     0.153  0.237   0.646
mid     0.131  0.270   0.485
high    0.235  0.389   0.604 

SSE 2010 onward
         mean    std  sharpe
bucket                      
low     0.076  0.165   0.460
mid    -0.002  0.186  -0.013
high   -0.002  0.228  -0.007


## 3.4 Real-time positions in SSE, 1996-2009

The real-time constant is first estimated on the extreme early-1990s data, which keeps positions very low.

In [6]:
print(w_sse.loc["1996":"2009"].describe().round(2))

count    168.00
mean       0.11
std        0.10
min        0.01
25%        0.04
50%        0.09
75%        0.16
max        0.59
Name: close, dtype: float64
